# M01A · Core Python: Variables, Data Structures, Control Flow & Functions (Solutions)
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M01_Python_Basics/solutions/M01A_Core_Python_Solutions.ipynb)

⏱ ~120 min · Difficulty: ★☆☆ · Prerequisites: M00_Setup


## 1. Learning Outcomes

By the end of this notebook, you will be able to:
1. **Declare and cast** primitive data types (`int`, `float`, `str`, `bool`) to represent sequence lengths, melting temperatures, and genomic annotations.
2. **Manipulate** biological sequences using zero-based indexing, slicing (`[start:stop:step]`), and built-in string methods.
3. **Construct and query** composite data structures (`list`, `tuple`, `set`, `dict`) using both loops and comprehensions.
4. **Implement** conditional branching (`if/elif/else`, ternary operators) to filter biological features, categorize nucleotides, and classify codons.
5. **Iterate** across sequences, codons, and homologous pairs using `for`, `while`, `range`, `enumerate`, and `zip`.
6. **Design** modular, reusable, documented functions with type hints, default arguments, and docstrings.
7. **Parse and extract** structured biological records from raw formats such as FASTA files and IUPAC motifs.


## 2. Prerequisites

- Completion of **M00_Setup_and_Orientation**.
- Basic understanding of DNA base pairing ($A, C, G, T$) and central dogma concepts.


## 3. Setup

Run this cell to detect the runtime environment, establish path references to data and output directories, and verify required modules.


In [ ]:
# --- Setup -------------------------------------------------------
import sys
import os
import platform
from pathlib import Path

# Detect Google Colab execution
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Cloning repository...")
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    p = Path.cwd()
    found_root = None
    for candidate in [p] + list(p.parents):
        if (candidate / "data" / "sequences").exists() or (candidate / "COURSE_PLAN.md").exists():
            found_root = candidate
            break
    if found_root:
        ROOT = found_root
    else:
        repo_candidate = Path("/Users/prateekpaul/Library/CloudStorage/GoogleDrive-prateekp@iiitd.ac.in/My Drive/IIITD/mygithub/ProgrammingCB")
        ROOT = repo_candidate if repo_candidate.exists() else p

DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / "M01A"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Python version     : {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"✓ Repository Root    : {ROOT.resolve()}")
print(f"✓ Data Directory     : {DATA_DIR.resolve()}")
print(f"✓ Output Directory   : {OUT_DIR.resolve()}")


## 4. Concept Sections

---

### 1A.1 Variables, Dynamic Typing & Type Casting

#### 4.1.1 Theory: Primitive Types in Bioinformatics
In Python, variables are named references pointing to objects in memory. You do not need to declare types explicitly—Python is dynamically typed.

| Python Type | Biological Meaning / Example | Typical Value |
|---|---|---|
| `int` | Sequence length, nucleotide count, chromosome number | `150`, `3200000000` |
| `float` | Melting temperature ($T_m$), GC content fraction, molecular weight | `58.4`, `0.485` |
| `str` | FASTA sequence, gene identifier, motif | `"ATGCGTAG"`, `"TP53"` |
| `bool` | Quality flag, coding status, forward strand boolean | `True`, `False` |

Type casting converts an object from one type to another: `int("120")`, `float("56.8")`, `str(42)`, `bool(1)`.


In [ ]:
# Worked Example: Modeling a PCR Primer's Properties
primer_name = "TP53_Exon4_Fwd"       # str
primer_seq = "ATGGAGGAGCCGCAGTCA"    # str
length_nt = len(primer_seq)          # int: 18 nucleotides
gc_fraction = 0.5555                # float: 55.6% GC
is_coding_strand = True              # bool

print("primer_seq type :", type(primer_seq))
print("length_nt type  :", type(length_nt))
print("gc_fraction type:", type(gc_fraction))

gc_percent_str = str(round(gc_fraction * 100, 1))
print(f"Primer {primer_name} ({length_nt} bp): GC% = {gc_percent_str}%")


#### 4.1.2 ✏️ Try it: Casting and Parsing Coordinates
Given raw coordinate strings from an external file: `start_str = "10450"` and `end_str = "10850"`, convert them to integers and compute the genomic span length (`end - start`).


In [ ]:
start_str = "10450"
end_str = "10850"

start_pos = int(start_str)
end_pos = int(end_str)
span_length = end_pos - start_pos

print(f"Genomic feature spans {span_length} base pairs (from {start_pos} to {end_pos}).")
assert span_length == 400
assert isinstance(span_length, int)


#### 4.1.3 ⚠️ Common Mistakes
- **Integer vs. Float Division**: In Python 3, `/` always yields a `float` (`10 / 2 -> 5.0`), while `//` performs floor division (`10 // 3 -> 3`).
- **Implicit string concatenation error**: Attempting `print("Length: " + length_nt)` raises `TypeError`. Use f-strings instead: `print(f"Length: {length_nt}")`.


---

### 1A.2 Operators: Arithmetic, Comparison, Logical & Membership

#### 4.2.1 Theory: Making Decisions on Biological Data
- **Arithmetic**: `+`, `-`, `*`, `/`, `//`, `%` (modulo: useful for reading frames!), `**`.
- **Comparison**: `==`, `!=`, `<`, `>`, `<=`, `>=`.
- **Logical**: `and`, `or`, `not`.
- **Membership**: `in`, `not in` — directly tests if a substring or element exists within a sequence or container.


In [ ]:
# Worked Example: Checking Promoter Sequence Quality
promoter = "GGCTATAAAAGGCCCGTCGA"

has_tata = "TATAAA" in promoter
is_valid_len = len(promoter) >= 20
is_triplet = (len(promoter) % 3) == 0
passes_qc = has_tata and is_valid_len

print(f"Promoter: {promoter}")
print(f"  Has TATA box : {has_tata}")
print(f"  Length >= 20 : {is_valid_len}")
print(f"  Length % 3==0: {is_triplet}")
print(f"  Passed QC    : {passes_qc}")


#### 4.2.2 ✏️ Try it: Ambiguous Base Check
Check whether the sequence `candidate_seq = "ATGCGRTACG"` contains degenerate nucleotide `'N'` or `'R'` using membership operators and boolean logic.


In [ ]:
candidate_seq = "ATGCGRTACG"
has_ambiguity = ('N' in candidate_seq) or ('R' in candidate_seq)
print(f"Sequence: {candidate_seq} | Contains ambiguity: {has_ambiguity}")
assert has_ambiguity is True


#### 4.2.3 ⚠️ Common Mistakes
- **Assignment (`=`) vs. Equality (`==`)**: Writing `if base = 'A':` is a syntax error.
- **Incorrect compound conditions**: Writing `if base == 'A' or 'G':` always evaluates to `True` because `'G'` is truthy! The correct expression is `if base in ('A', 'G'):`.


---

### 1A.3 String Manipulation & Slicing

#### 4.3.1 Theory: The Grammar of Biological Sequences
In computational biology, DNA, RNA, and protein molecules are represented as strings.
Python uses **0-based indexing**:
```
Index:    0   1   2   3   4   5   6   7   8   9  10  11
Base:     A   T   G   C   G   T   A   C   G   T   T   A
Neg idx: -12 -11 -10  -9  -8  -7  -6  -5  -4  -3  -2  -1
```

**Slicing syntax**: `sequence[start:stop:step]`
- `start`: inclusive beginning index (defaults to 0).
- `stop`: **exclusive** end index.
- `step`: stride / increment.
- Reversing a sequence: `sequence[::-1]`.


In [ ]:
# Worked Example: Slicing Codons and Modifying Sequences
dna = "ATGCGTACTGAC"

start_codon = dna[0:3]
stop_codon = dna[-3:]
first_base_of_codons = dna[0::3]
reversed_dna = dna[::-1]
mrna = dna.replace("T", "U")

print(f"Original DNA : {dna}")
print(f"Start codon  : {start_codon}")
print(f"Stop codon   : {stop_codon}")
print(f"Codon 1st pos: {first_base_of_codons}")
print(f"Reversed DNA : {reversed_dna}")
print(f"mRNA transcript: {mrna}")


#### 4.3.2 ✏️ Try it: Slicing the 2nd Codon
Extract the 2nd codon (indices `3:6` in 0-based Python) from `seq = "ATGCGTAAATAG"`.


In [ ]:
seq = "ATGCGTAAATAG"
second_codon = seq[3:6]
print("Second codon:", second_codon)
assert second_codon == "CGT"


#### 4.3.3 ⚠️ Common Mistakes
- **Strings are immutable**: `seq[0] = "G"` raises `TypeError`. You must create a new string using slicing or `.replace()`.
- **Off-by-one errors**: `seq[0:3]` contains 3 characters (indices 0, 1, and 2). Index 3 is excluded.


---

### 1A.4 Lists & List Comprehensions

#### 4.4.1 Theory: Ordered, Mutable Collections
A `list` is an ordered, mutable sequence of items. In bioinformatics, lists store collections of sequencing reads, codons, mutation positions, or gene IDs.

**List Comprehensions**:
```python
[expression for item in iterable if condition]
```


In [ ]:
# Worked Example: Splitting a Gene into Codons using List Comprehension
cds = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"

codons = [cds[i:i+3] for i in range(0, len(cds), 3) if len(cds[i:i+3]) == 3]
coding_codons = [c for c in codons if c not in ("TAA", "TAG", "TGA")]

print(f"Total codons identified : {len(codons)}")
print(f"First 5 codons          : {codons[:5]}")
print(f"Coding codons (no stops): {len(coding_codons)} codons")


#### 4.4.2 ✏️ Try it: Filtering Long Sequences
Given `lengths = [45, 150, 75, 250, 30, 300, 120]`, use a list comprehension to extract lengths >= 100.


In [ ]:
lengths = [45, 150, 75, 250, 30, 300, 120]
long_reads = [l for l in lengths if l >= 100]
print("Long reads:", long_reads)
assert long_reads == [150, 250, 300, 120]


---

### 1A.5 Tuples & Sets

#### 4.5.1 Theory: Immutability and Unique Elements
- **Tuples** (`tuple`): Ordered, immutable sequences `(1, 2, 3)`. Ideal for fixed genomic coordinates: `("chr7", 140453136, 140453193, "+")`.
- **Sets** (`set`): Unordered collections of unique elements `{1, 2, 3}`. Membership testing (`x in my_set`) operates in $O(1)$ constant time.


In [ ]:
# Worked Example: Comparing K-mer Motifs Between Two Sequences
seq_a = "ATGCGTAGCTAGCT"
seq_b = "CGTAGCTAGCTTTT"
k = 4

kmers_a = {seq_a[i:i+k] for i in range(len(seq_a) - k + 1)}
kmers_b = {seq_b[i:i+k] for i in range(len(seq_b) - k + 1)}

shared_kmers = kmers_a & kmers_b       # Intersection
unique_to_a = kmers_a - kmers_b        # Difference
all_kmers = kmers_a | kmers_b          # Union

print(f"4-mers in Seq A: {kmers_a}")
print(f"4-mers in Seq B: {kmers_b}")
print(f"Shared 4-mers  : {shared_kmers}")
print(f"Unique to Seq A: {unique_to_a}")


#### 4.5.2 ✏️ Try it: Unique Amino Acids
Extract the set of unique amino acids from `peptide = "MKWVTFISLLLLFSSAYSRG"`.


In [ ]:
peptide = "MKWVTFISLLLLFSSAYSRG"
unique_aa = set(peptide)
print("Unique amino acids:", sorted(unique_aa))
assert "L" in unique_aa
assert len(unique_aa) == 13


---

### 1A.6 Dictionaries & Hash Maps

#### 4.6.1 Theory: Key-Value Associations in Biology
A `dict` associates unique keys with values with $O(1)$ lookup time.
Key methods: `d.get(k, default)`, `d.items()`, `d.keys()`, `d.values()`.


In [ ]:
# Worked Example: Genetic Code Codon Table Mapping
codon_table = {
    "ATG": "M", "TGG": "W", "TTT": "F", "TTC": "F",
    "GAA": "E", "GAG": "E", "TAA": "*", "TAG": "*", "TGA": "*"
}

sample_codons = ["ATG", "TTT", "GAG", "NNN", "TAG"]
translated_aa = [codon_table.get(c, "X") for c in sample_codons]

print("Sample codons:", sample_codons)
print("Translation  :", translated_aa)

dna_sample = "ATGCGTACGTTTGCAACCG"
base_counts = {base: dna_sample.count(base) for base in "ACGT"}
print("Base counts  :", base_counts)


#### 4.6.2 ✏️ Try it: Dinucleotide Frequency
Given `seq = "ACACAC"`, count occurrences of `"AC"` and `"CA"`.


In [ ]:
seq = "ACACAC"
dinc_counts = {"AC": seq.count("AC"), "CA": seq.count("CA")}
print("Dinucleotide counts:", dinc_counts)
assert dinc_counts["AC"] == 3
assert dinc_counts["CA"] == 2


---

### 1A.7 Conditional Statements & Branching Logic

#### 4.7.1 Theory: Decision Structures
Biological scripts constantly filter data based on biochemical properties using `if/elif/else`.


In [ ]:
# Worked Example: Nucleotide Chemical Classification
def classify_nucleotide(base: str) -> str:
    base = base.upper()
    if base in ("A", "G"):
        return "Purine (double ring)"
    elif base in ("C", "T", "U"):
        return "Pyrimidine (single ring)"
    elif base == "N":
        return "Ambiguous (unknown base)"
    else:
        return "Invalid character"

for b in ["A", "c", "T", "N", "Z"]:
    print(f"Base '{b}' -> {classify_nucleotide(b)}")


#### 4.7.2 ✏️ Try it: Classify Amino Acid by Charge
Write an `if/elif/else` expression to classify an amino acid `aa` as `"Positive"`, `"Negative"`, or `"Neutral"`.


In [ ]:
def get_charge_class(aa: str) -> str:
    aa = aa.upper()
    if aa in ("R", "K", "H"):
        return "Positive"
    elif aa in ("D", "E"):
        return "Negative"
    else:
        return "Neutral"

assert get_charge_class("K") == "Positive"
assert get_charge_class("d") == "Negative"
assert get_charge_class("A") == "Neutral"
print("✓ Charge check passed!")


---

### 1A.8 Loops & Iteration Patterns

#### 4.8.1 Theory: Iteration Across Sequences and Pairs
- `for item in iterable`: standard iteration.
- `range(start, stop, step)`: generate numeric sequences.
- `enumerate(iterable, start=0)`: yields `(index, item)` pairs.
- `zip(seq1, seq2)`: iterates across two parallel sequences.


In [ ]:
# Worked Example: Comparing Homologous Reads using zip() and enumerate()
seq1 = "ATGCGTACGT"
seq2 = "ATACGTTCGT"

mismatches = []
for idx, (b1, b2) in enumerate(zip(seq1, seq2)):
    if b1 != b2:
        mismatches.append((idx, b1, b2))
        print(f"  Position {idx}: {b1} -> {b2} (mismatch)")

print(f"Total mismatch count: {len(mismatches)}")


#### 4.8.2 ✏️ Try it: Walking Codons with range()
Print each codon and its 0-based start index in `seq = "ATGAAATAG"` using `range(0, len(seq), 3)`.


In [ ]:
seq = "ATGAAATAG"
codon_list = []
for i in range(0, len(seq), 3):
    codon = seq[i:i+3]
    codon_list.append((i, codon))
print("Codon list:", codon_list)
assert codon_list == [(0, "ATG"), (3, "AAA"), (6, "TAG")]


---

### 1A.9 Functions & Modular Code Design

#### 4.9.1 Theory: Reusable Bioinformatics Functions
Functions encapsulate logic, prevent code duplication, and enable automated testing.


In [ ]:
# Worked Example: Professional Bioinformatics Function
def calculate_gc_content(dna_sequence: str) -> float:
    """Calculates GC percentage of a DNA sequence."""
    if not dna_sequence:
        return 0.0
    clean_seq = dna_sequence.upper()
    gc_total = clean_seq.count("G") + clean_seq.count("C")
    return (gc_total / len(clean_seq)) * 100.0

print("GC% of 'ATGC':", calculate_gc_content("ATGC"))
assert calculate_gc_content("ATGC") == 50.0
assert calculate_gc_content("GGCC") == 100.0
assert calculate_gc_content("") == 0.0


#### 4.9.2 ✏️ Try it: Base Counting Function
Write a function `count_bases(seq: str) -> dict[str, int]` that returns counts for `'A'`, `'C'`, `'G'`, and `'T'`.


In [ ]:
def count_bases(seq: str) -> dict[str, int]:
    clean = seq.upper()
    return {base: clean.count(base) for base in "ACGT"}

counts = count_bases("ATGCATGC")
assert counts == {"A": 2, "C": 2, "G": 2, "T": 2}


## 5. Practice Exercises

Solve each of the 20 problems below.


### Exercise 1 (Level 1 ★): Count Guanine Residues in a DNA Sequence

**Problem**: Write a function `count_g(dna_sequence: str) -> int` that counts how many Guanine ('G' or 'g') residues are present.

- **Input**: String `dna_sequence`.
- **Expected Output**: Integer count of 'G'/'g'.


In [ ]:
def count_g(dna_sequence: str) -> int:
    """Counts Guanine bases (case-insensitive)."""
    return dna_sequence.upper().count('G')


#### 💡 Solution Explanation
Normalizes to uppercase and calls the C-optimized `.count('G')` method ($O(N)$ time).

In [ ]:
assert count_g('ATGCGTACGTT') == 3
assert count_g('aaaaattttt') == 0
print('✓ Exercise 1 passed!')


### Exercise 2 (Level 1 ★): Extract First and Last Codon

**Problem**: Write a function `first_and_last_codon(seq: str) -> tuple[str, str]` that extracts the first and last 3 nucleotides.

- **Input**: String `seq` of length >= 6.
- **Expected Output**: Tuple `(first_codon, last_codon)` in uppercase.


In [ ]:
def first_and_last_codon(seq: str) -> tuple[str, str]:
    clean = seq.upper()
    return (clean[:3], clean[-3:])


#### 💡 Solution Explanation
Slices `clean[:3]` for the first codon and `clean[-3:]` for the terminal codon.

In [ ]:
assert first_and_last_codon('ATGCGTACGTTTAA') == ('ATG', 'TAA')
print('✓ Exercise 2 passed!')


### Exercise 3 (Level 1 ★): Nucleotide Ambiguity Checker

**Problem**: Write a function `has_ambiguous_bases(seq: str) -> bool` returning `True` if any character is NOT one of `A, C, G, T`.

- **Input**: String `seq`.
- **Expected Output**: Boolean `True` or `False`.


In [ ]:
def has_ambiguous_bases(seq: str) -> bool:
    valid = set('ACGT')
    return any(c not in valid for c in seq.upper())


#### 💡 Solution Explanation
Uses a set of canonical nucleotides with short-circuit evaluation via `any()`.

In [ ]:
assert has_ambiguous_bases('ATGCGT') is False
assert has_ambiguous_bases('ATGCNGT') is True
print('✓ Exercise 3 passed!')


### Exercise 4 (Level 1 ★): Amino Acid Code Converter

**Problem**: Write a function `aa_to_three_letter(code: str) -> str` mapping 1-letter amino acid code to standard 3-letter abbreviation (or 'Unknown').

- **Input**: Single character string `code`.
- **Expected Output**: 3-letter abbreviation string.


In [ ]:
def aa_to_three_letter(code: str) -> str:
    aa_map = {'A':'Ala','R':'Arg','N':'Asn','D':'Asp','C':'Cys','E':'Glu','Q':'Gln','G':'Gly','H':'His','I':'Ile','L':'Leu','K':'Lys','M':'Met','F':'Phe','P':'Pro','S':'Ser','T':'Thr','W':'Trp','Y':'Tyr','V':'Val'}
    return aa_map.get(code.upper(), 'Unknown')


#### 💡 Solution Explanation
Dictionary lookup using `.get(key, 'Unknown')` avoids `KeyError` on unexpected inputs.

In [ ]:
assert aa_to_three_letter('M') == 'Met'
assert aa_to_three_letter('w') == 'Trp'
assert aa_to_three_letter('X') == 'Unknown'
print('✓ Exercise 4 passed!')


### Exercise 5 (Level 1 ★): Classify Nucleotide as Purine or Pyrimidine

**Problem**: Write a function `classify_base(base: str) -> str` returning 'Purine' for A/G, 'Pyrimidine' for C/T/U, and 'Other' otherwise.

- **Input**: Single character string `base`.
- **Expected Output**: 'Purine', 'Pyrimidine', or 'Other'.


In [ ]:
def classify_base(base: str) -> str:
    b = base.upper()
    if b in ('A', 'G'): return 'Purine'
    elif b in ('C', 'T', 'U'): return 'Pyrimidine'
    return 'Other'


#### 💡 Solution Explanation
Classifies heterocyclic bases by chemical ring size (purine = 2 rings, pyrimidine = 1 ring).

In [ ]:
assert classify_base('A') == 'Purine'
assert classify_base('c') == 'Pyrimidine'
assert classify_base('N') == 'Other'
print('✓ Exercise 5 passed!')


### Exercise 6 (Level 2 ★★): Split Sequence into Non-Overlapping Codons

**Problem**: Write a function `get_codons(dna_sequence: str) -> list[str]` splitting sequence into 3-nt codons, discarding incomplete tails.

- **Input**: String `dna_sequence`.
- **Expected Output**: List of 3-letter strings.


In [ ]:
def get_codons(dna_sequence: str) -> list[str]:
    clean = dna_sequence.upper()
    num = len(clean) // 3
    return [clean[i*3:(i+1)*3] for i in range(num)]


#### 💡 Solution Explanation
Integer division `len // 3` calculates full triplets, sliced with step stride.

In [ ]:
assert get_codons('ATGCGTACGTT') == ['ATG', 'CGT', 'ACG']
print('✓ Exercise 6 passed!')


### Exercise 7 (Level 2 ★★): Base Frequencies Using Dict Comprehension

**Problem**: Write a function `base_frequencies(seq: str) -> dict[str, int]` returning counts of A, C, G, T ensuring all 4 keys exist.

- **Input**: String `seq`.
- **Expected Output**: Dictionary mapping 'A','C','G','T' to counts.


In [ ]:
def base_frequencies(seq: str) -> dict[str, int]:
    clean = seq.upper()
    return {b: clean.count(b) for b in 'ACGT'}


#### 💡 Solution Explanation
Dictionary comprehension iterating over fixed 'ACGT' guarantees all 4 canonical keys.

In [ ]:
assert base_frequencies('ATGC') == {'A': 1, 'C': 1, 'G': 1, 'T': 1}
print('✓ Exercise 7 passed!')


### Exercise 8 (Level 2 ★★): Transitions vs Transversions Counter

**Problem**: Write a function `count_transitions_transversions(s1: str, s2: str) -> tuple[int, int]` that compares two equal-length sequences and counts transitions ($A \leftrightarrow G, C \leftrightarrow T$) and transversions.

- **Input**: Two equal-length strings.
- **Expected Output**: Tuple `(transitions, transversions)`.


In [ ]:
def count_transitions_transversions(s1: str, s2: str) -> tuple[int, int]:
    if len(s1) != len(s2): raise ValueError("Unequal lengths!")
    purines = {'A', 'G'}
    pyrimidines = {'C', 'T'}
    ts, tv = 0, 0
    for b1, b2 in zip(s1.upper(), s2.upper()):
        if b1 != b2:
            if (b1 in purines and b2 in purines) or (b1 in pyrimidines and b2 in pyrimidines):
                ts += 1
            else:
                tv += 1
    return (ts, tv)


#### 💡 Solution Explanation
Compares aligned pairs with `zip()`, testing ring conservation.

In [ ]:
assert count_transitions_transversions('ACGT', 'GAGT') == (1, 1)
print('✓ Exercise 8 passed!')


### Exercise 9 (Level 2 ★★): Reverse Sequence Without Slicing

**Problem**: Write a function `reverse_sequence(seq: str) -> str` that reverses a string without using `[::-1]`.

- **Input**: String `seq`.
- **Expected Output**: Reversed string.


In [ ]:
def reverse_sequence(seq: str) -> str:
    return ''.join(reversed(seq))


#### 💡 Solution Explanation
Uses `reversed()` iterator and joins characters into a new string.

In [ ]:
assert reverse_sequence('ATGC') == 'CGTA'
print('✓ Exercise 9 passed!')


### Exercise 10 (Level 2 ★★): Extract Unique K-mers

**Problem**: Write a function `unique_kmers(seq: str, k: int) -> set[str]` extracting all unique k-mers.

- **Input**: String `seq` and integer `k`.
- **Expected Output**: Set of unique k-mer strings.


In [ ]:
def unique_kmers(seq: str, k: int) -> set[str]:
    clean = seq.upper()
    if len(clean) < k or k <= 0: return set()
    return {clean[i:i+k] for i in range(len(clean) - k + 1)}


#### 💡 Solution Explanation
Sliding window with set comprehension automatically deduplicates k-mers.

In [ ]:
assert unique_kmers('ACACAC', 2) == {'AC', 'CA'}
print('✓ Exercise 10 passed!')


### Exercise 11 (Level 2 ★★): Safe Codon Translation with Fallback

**Problem**: Write a function `translate_codon(codon: str, table: dict[str, str]) -> str` translating a codon safely, returning 'X' if missing.

- **Input**: String `codon` and dictionary `table`.
- **Expected Output**: Single amino acid character.


In [ ]:
def translate_codon(codon: str, table: dict[str, str]) -> str:
    return table.get(codon.upper(), 'X')


#### 💡 Solution Explanation
Safe query via `.get(key, 'X')` prevents `KeyError` crashes.

In [ ]:
t = {'ATG':'M'}
assert translate_codon('ATG', t) == 'M'
assert translate_codon('NNN', t) == 'X'
print('✓ Exercise 11 passed!')


### Exercise 12 (Level 3 ★★★): Find All Motif Positions (With Overlaps)

**Problem**: Write a function `find_motif_positions(seq: str, motif: str) -> list[int]` finding all 0-based start positions allowing overlaps.

- **Input**: Strings `seq` and `motif`.
- **Expected Output**: List of start indices.


In [ ]:
def find_motif_positions(seq: str, motif: str) -> list[int]:
    s = seq.upper()
    m = motif.upper()
    if not m or len(s) < len(m): return []
    pos = []
    start = 0
    while True:
        idx = s.find(m, start)
        if idx == -1: break
        pos.append(idx)
        start = idx + 1
    return pos


#### 💡 Solution Explanation
Advances `start = idx + 1` upon each find, ensuring overlapping occurrences are detected.

In [ ]:
assert find_motif_positions('ATATATA', 'ATA') == [0, 2, 4]
print('✓ Exercise 12 passed!')


### Exercise 13 (Level 3 ★★★): Dinucleotide Composition Matrix

**Problem**: Write a function `dinucleotide_matrix(seq: str) -> dict[str, dict[str, int]]` creating a 4x4 nested dictionary of dinucleotide counts.

- **Input**: String `seq`.
- **Expected Output**: Nested dictionary of counts.


In [ ]:
def dinucleotide_matrix(seq: str) -> dict[str, dict[str, int]]:
    bases = 'ACGT'
    matrix = {b1: {b2: 0 for b2 in bases} for b1 in bases}
    clean = seq.upper()
    for i in range(len(clean) - 1):
        d = clean[i:i+2]
        if d[0] in bases and d[1] in bases:
            matrix[d[0]][d[1]] += 1
    return matrix


#### 💡 Solution Explanation
Initializes complete 16-element nested dictionary and increments counts with a 2-bp sliding window.

In [ ]:
res = dinucleotide_matrix('ACACAC')
assert res['A']['C'] == 3
assert res['C']['A'] == 2
print('✓ Exercise 13 passed!')


### Exercise 14 (Level 3 ★★★): Sliding Window GC Content Calculator

**Problem**: Write a function `sliding_window_gc(seq: str, window_size: int, step: int) -> list[float]` computing GC% in sliding windows.

- **Input**: String `seq`, integers `window_size`, `step`.
- **Expected Output**: List of float GC percentages.


In [ ]:
def sliding_window_gc(seq: str, window_size: int, step: int) -> list[float]:
    clean = seq.upper()
    res = []
    for i in range(0, len(clean) - window_size + 1, step):
        w = clean[i:i+window_size]
        gc = w.count('G') + w.count('C')
        res.append(round((gc / window_size) * 100.0, 2))
    return res


#### 💡 Solution Explanation
Strided range iteration extracts windows, computes GC percentage, and rounds to 2 decimals.

In [ ]:
assert sliding_window_gc('GCGCATATGCGC', 4, 4) == [100.0, 0.0, 100.0]
print('✓ Exercise 14 passed!')


### Exercise 15 (Level 3 ★★★): Clean and Validate Sequence Batch

**Problem**: Write a function `clean_and_validate_batch(raw_sequences: list[str]) -> tuple[list[str], list[str]]` partitioning into valid and invalid.

- **Input**: List of strings.
- **Expected Output**: Tuple `(valid_seqs, invalid_seqs)`.


In [ ]:
def clean_and_validate_batch(raw_sequences: list[str]) -> tuple[list[str], list[str]]:
    valid_b = set('ACGT')
    valid, invalid = [], []
    for r in raw_sequences:
        c = ''.join(r.split()).upper()
        if c and all(b in valid_b for b in c):
            valid.append(c)
        else:
            invalid.append(c if c else r)
    return (valid, invalid)


#### 💡 Solution Explanation
Strips whitespace and verifies characters against canonical alphabet, appending to respective partitions.

In [ ]:
v, inv = clean_and_validate_batch([' atg cgt ', 'ATGC123', 'ccgta'])
assert v == ['ATGCGT', 'CCGTA']
print('✓ Exercise 15 passed!')


### Exercise 16 (Level 1 ★): Amino Acid Residue Mass Lookup

**Problem**: Write a function `aa_residue_mass(aa: str) -> float` returning monoisotopic residue mass in Daltons (e.g. G: 57.02, A: 71.04, W: 186.08) or 0.0 if unknown.

- **Input**: Single character string `aa`.
- **Expected Output**: Float mass in Daltons.


In [ ]:
def aa_residue_mass(aa: str) -> float:
    masses = {
        'A': 71.04, 'R': 156.10, 'N': 114.04, 'D': 115.03, 'C': 103.01,
        'E': 129.04, 'Q': 128.06, 'G': 57.02, 'H': 137.06, 'I': 113.08,
        'L': 113.08, 'K': 128.09, 'M': 131.04, 'F': 147.07, 'P': 97.05,
        'S': 87.03, 'T': 101.05, 'W': 186.08, 'Y': 163.06, 'V': 99.07
    }
    return masses.get(aa.upper(), 0.0)


#### 💡 Solution Explanation
Dictionary lookup of monoisotopic dehydrated amino acid residue masses used in mass spectrometry.

In [ ]:
assert aa_residue_mass('G') == 57.02
assert aa_residue_mass('W') == 186.08
assert aa_residue_mass('Z') == 0.0
print('✓ Exercise 16 passed!')


### Exercise 17 (Level 2 ★★): CpG Island Observed-to-Expected Ratio

**Problem**: Write a function `cpg_ratio(dna_seq: str) -> float` calculating observed/expected CpG ratio: $(\text{count}(CG) \times N) / (\text{count}(C) \times \text{count}(G))$. Round to 3 decimals.

- **Input**: DNA string `dna_seq`.
- **Expected Output**: Float ratio.


In [ ]:
def cpg_ratio(dna_seq: str) -> float:
    clean = dna_seq.upper()
    n = len(clean)
    if n < 2: return 0.0
    c_c = clean.count('C')
    g_c = clean.count('G')
    if c_c == 0 or g_c == 0: return 0.0
    cg = sum(1 for i in range(n - 1) if clean[i:i+2] == 'CG')
    expected = (c_c * g_c) / n
    return round(cg / expected, 3)


#### 💡 Solution Explanation
Calculates CpG ratio, a hallmark metric for detecting vertebrate gene promoters.

In [ ]:
assert cpg_ratio('CGCG') == 2.0
assert cpg_ratio('AAAA') == 0.0
print('✓ Exercise 17 passed!')


### Exercise 18 (Level 2 ★★): Codon Usage Frequency Table

**Problem**: Write a function `codon_usage_frequencies(cds: str) -> dict[str, float]` returning relative percentage of each codon among full codons, rounded to 2 decimals.

- **Input**: CDS string.
- **Expected Output**: Dictionary mapping `{codon: percentage}`.


In [ ]:
def codon_usage_frequencies(cds: str) -> dict[str, float]:
    clean = cds.upper()
    n = len(clean) // 3
    if n == 0: return {}
    codons = [clean[i*3:(i+1)*3] for i in range(n)]
    counts = {}
    for c in codons: counts[c] = counts.get(c, 0) + 1
    return {c: round((cnt / n) * 100.0, 2) for c, cnt in counts.items()}


#### 💡 Solution Explanation
Converts triplet frequencies to relative codon usage percentages.

In [ ]:
res = codon_usage_frequencies('ATGGCCATGTAA')
assert res['ATG'] == 50.0
assert res['GCC'] == 25.0
print('✓ Exercise 18 passed!')


### Exercise 19 (Level 3 ★★★): IUPAC Degenerate Motif Matching Without Regex

**Problem**: Write a function `match_iupac_motif(seq: str, iupac_motif: str) -> list[int]` finding all 0-based start positions in `seq` matching degenerate IUPAC motif (R=A/G, Y=C/T, S=G/C, W=A/T, N=any).

- **Input**: Strings `seq` and `iupac_motif`.
- **Expected Output**: List of start indices.


In [ ]:
def match_iupac_motif(seq: str, iupac_motif: str) -> list[int]:
    iupac_map = {
        'A': {'A'}, 'C': {'C'}, 'G': {'G'}, 'T': {'T'},
        'R': {'A', 'G'}, 'Y': {'C', 'T'}, 'S': {'G', 'C'},
        'W': {'A', 'T'}, 'N': {'A', 'C', 'G', 'T'}
    }
    s, m = seq.upper(), iupac_motif.upper()
    k = len(m)
    matches = []
    for i in range(len(s) - k + 1):
        win = s[i:i+k]
        if all(char_s in iupac_map.get(char_m, set()) for char_s, char_m in zip(win, m)):
            matches.append(i)
    return matches


#### 💡 Solution Explanation
Checks IUPAC set compatibility character-by-character along a sliding window.

In [ ]:
assert match_iupac_motif('ACGATGTTT', 'RYS') == [0, 3]
print('✓ Exercise 19 passed!')


### Exercise 20 (Level 3 ★★★): Multi-Record FASTA Parser

**Problem**: Write a function `parse_fasta_records(fasta_text: str) -> list[dict]` parsing a multi-line FASTA string into `[{'id': str, 'description': str, 'sequence': str, 'length': int}, ...]`. First word of header is id, rest is description.

- **Input**: Multi-line FASTA string.
- **Expected Output**: List of record dictionaries.


In [ ]:
def parse_fasta_records(fasta_text: str) -> list[dict]:
    records = []
    curr_id, curr_desc, parts = None, "", []
    for line in fasta_text.strip().splitlines():
        line = line.strip()
        if not line: continue
        if line.startswith(">"):
            if curr_id is not None:
                seq = "".join(parts).upper()
                records.append({"id": curr_id, "description": curr_desc, "sequence": seq, "length": len(seq)})
            header = line[1:].strip().split(maxsplit=1)
            curr_id = header[0]
            curr_desc = header[1] if len(header) > 1 else ""
            parts = []
        else:
            parts.append(line)
    if curr_id is not None:
        seq = "".join(parts).upper()
        records.append({"id": curr_id, "description": curr_desc, "sequence": seq, "length": len(seq)})
    return records


#### 💡 Solution Explanation
Stateful parser handling multiline sequences and header fields.

In [ ]:
fasta_data = ">seq1 TP53\nATGC\nGTAG\n>seq2 Trp53\nCCGG"
r = parse_fasta_records(fasta_data)
assert len(r) == 2
assert r[0]["id"] == "seq1"
assert r[0]["sequence"] == "ATGCGTAG"
assert r[0]["length"] == 8
print('✓ Exercise 20 passed!')


## 6. Summary & Cheat Sheet Table

| Construct | Syntax | Example | Description |
|---|---|---|---|
| **Indexing** | `seq[i]` | `seq[0]`, `seq[-1]` | 0-based first base, negative last base |
| **Slicing** | `seq[start:stop:step]` | `seq[0:3]`, `seq[::-1]` | Extract range; reverse |
| **Methods** | `s.method()` | `s.upper()`, `s.count("G")` | Case normalization, frequency counting |
| **Membership** | `item in container` | `"ATG" in seq` | $O(1)$ set/dict check, substring check |
| **List Comp** | `[expr for x in it if c]` | `[c for c in codons if c != "TAA"]`| Transform and filter lists concisely |
| **Set Ops** | `s1 & s2`, `s1 | s2` | `kmers_a & kmers_b` | Fast intersection, union, difference |
| **Dict Get** | `d.get(k, default)` | `table.get(codon, "X")` | Safe retrieval avoiding `KeyError` |
| **Zip Iter** | `zip(seq1, seq2)` | `for b1, b2 in zip(s1, s2):` | Parallel tandem nucleotide iteration |
| **Enumerate** | `enumerate(it, start)` | `for i, b in enumerate(seq):` | Simultaneous track of index and element |


## 7. Further Reading & References

1. **Python Official Documentation**: *Data Structures Tutorial*. https://docs.python.org/3/tutorial/datastructures.html
2. **Real Python**: *Python Strings and Character Data*. https://realpython.com/python-strings/
3. **Rosalind Bioinformatics**: *Python Village Exercises (INI1 to INI6)*. https://rosalind.info/problems/list-view/?category=python-village
4. **PEP 257**: *Docstring Conventions*. https://peps.python.org/pep-0257/
